In [1]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import requests
from dotenv import load_dotenv
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer

c:\Users\Himanshu\Desktop\CampusAI\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
## Reading documents from the file system
directory_path = "./Web_Scrapper/"
loader = DirectoryLoader(
    path=directory_path,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}, # Add this line to specify encoding
    show_progress=True
)
docs = loader.load()
print(len(docs), "documents loaded.")
print(docs)

100%|██████████| 1/1 [00:00<00:00, 654.85it/s]

1 documents loaded.
[Document(metadata={'source': 'Web_Scrapper\\gehu-top-placement.txt'}, page_content='Top Placement Records\n \nStudent Name\tBranch\t  Company Name\tSalary Package (Lacs)\nDarshit Joshi\tComputer Science & Engineering\tAmazon\t47.88\nIkshit Dhyani\tComputer Science & Engineering\tAmazon\t47.88\nDivyansh Bhadra\tComputer Science & Engineering\tAmazon\t47.88\nVaibhav Gaur\tComputer Science & Engineering\tAmazon\t47.88\nChandni Mishra\tCS (GEN)\tAmazon\t47.88\nShreya Sharma\tCS (ML & AI)\tAmazon\t47.88\nPawandeep Kaur\tCS (GEN)\tIntuit\t44.92\nEpsita Mukherjee\tCS (GEN)\tPaypal\t34.40\nShreya Sharma\tCS (ML & AI)\tPaypal\t34.40\nPrakhar Dhyani\tCS (GEN)\tPaypal\t34.40\nSachin Upadhyay\tComputer Science & Engineering\tPaypal\t32.90\nSejal Aggarwal\tComputer Science & Engineering\tPaypal\t32.90\nIkshit Dhyani\tComputer Science & Engineering\tVisa\t32.88\nAbhinav\tComputer Science & Engineering\tVISA\t32.88\nPawandeep Kaur\tCS (GEN)\tFlipkart\t32.57\nKomal Kumari\tCS (GEN

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1)
chunk = text_splitter.split_documents(docs)
print(len(chunk), "chunks created.")

1 chunks created.


In [9]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
text = [doc.page_content for doc in chunk]
embeddings = model.encode(
    text,
    batch_size=32,
    show_progress_bar=True
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 542.31it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.68it/s]


In [10]:
print(len(embeddings), "embeddings created.")
print("Dimension of each embedding:", len(embeddings[0]))
print("Sample embedding:", embeddings[0][:5])
print(len(chunk), "chunks and", len(embeddings), "embeddings should match.")

1 embeddings created.
Dimension of each embedding: 384
Sample embedding: [-0.01081951 -0.03685579 -0.0045461  -0.0154393   0.02500464]
1 chunks and 1 embeddings should match.


In [11]:
client = MongoClient(os.getenv("MONGO_URI"))

db = client["campus_ai"]
collection = db["academic_vector"]
vectors = []

for i, embedding in enumerate(embeddings):
    vectors.append({
        "id": f"chunk-{i}",
        "values": embedding.tolist(),
        "metadata": {
            "text": chunk[i].page_content,
            "source": chunk[i].metadata.get("source"),
            "page": chunk[i].metadata.get("page")
        }
    })
collection.insert_many(vectors)

InsertManyResult([ObjectId('69a5d20779fe0bb2f1b509ea')], acknowledged=True)

In [15]:
query_vector = model.encode(["tell about top placements records in gehu"])[0].tolist()

pipeline = [
    {
        "$vectorSearch": {
            "index": "academic_index",
            "path": "values",
            "queryVector": query_vector,
            "numCandidates": 100,
            "limit": 5
        }
    },
    {
            "$project": {
                "_id": 0,
                "id": 1,
                "metadata": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
]

results = list(collection.aggregate(pipeline))
# print(results)
for doc in results:
    print(doc["metadata"]["text"])
    print(doc["score"])

URL: https://gehu.ac.in/
TITLE: Best University in Uttarakhand, India | Top Colleges - GEHU
0.8000993728637695
URL: https://gehu.ac.in/
TITLE: Best University in Uttarakhand, India | Top Colleges - GEHU
0.8000993728637695
Top Placement Records
 
Student Name	Branch	  Company Name	Salary Package (Lacs)
Darshit Joshi	Computer Science & Engineering	Amazon	47.88
Ikshit Dhyani	Computer Science & Engineering	Amazon	47.88
Divyansh Bhadra	Computer Science & Engineering	Amazon	47.88
Vaibhav Gaur	Computer Science & Engineering	Amazon	47.88
Chandni Mishra	CS (GEN)	Amazon	47.88
Shreya Sharma	CS (ML & AI)	Amazon	47.88
Pawandeep Kaur	CS (GEN)	Intuit	44.92
Epsita Mukherjee	CS (GEN)	Paypal	34.40
Shreya Sharma	CS (ML & AI)	Paypal	34.40
Prakhar Dhyani	CS (GEN)	Paypal	34.40
Sachin Upadhyay	Computer Science & Engineering	Paypal	32.90
Sejal Aggarwal	Computer Science & Engineering	Paypal	32.90
Ikshit Dhyani	Computer Science & Engineering	Visa	32.88
Abhinav	Computer Science & Engineering	VISA	32.88
Pawandeep